# 05 · RQ4: forecast uncertainty and q90-based warning

LightGBM quantile models (alpha 0.1 / 0.5 / 0.9) give a [q10, q90] interval per horizon; a single widening factor is fitted on VALIDATION when the coverage is clearly off the 0.80 target. The test split is evaluated once. The q90 forecast then replaces the point forecast in the RQ2 warning rule. Run notebooks 02 and 03 first.

*Estimated time: QUICK: 2-4 min.*

In [ ]:
# --- parameters -------------------------------------------------------------------------
QUICK = True        # 2 stations, short run (< 5 min); set False for the full 12-station run
USE_DRIVE = False   # Colab only: mount Google Drive to keep results between sessions
RUN_OPTUNA = False  # hyper-parameter search (notebook 02 onwards)
SEEDS = [0]


In [ ]:
# --- setup (runs on Colab, skipped locally) ---------------------------------------------
import os
import subprocess
import sys
IN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/furkantopcuu/airq-iot.git"
if IN_COLAB:
    if not os.path.exists("/content/airq-iot"):
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/airq-iot"], check=True)
    os.chdir("/content/airq-iot")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml]"], check=True)
    sys.path.insert(0, "/content/airq-iot/src")  # no restart needed after the editable install


In [ ]:
from airq.config import load_config
from airq.env import mount_drive_if_wanted
from airq.pipelines.colab import env_summary

mount_drive_if_wanted(USE_DRIVE)
cfg = load_config()
env_summary(cfg)


In [ ]:
from airq.pipelines.rq4 import run_uncertainty

summary = run_uncertainty(cfg, quick=QUICK, force=QUICK)  # full run: guarded, test is seen once
summary["scales"]


In [ ]:
import pandas as pd
from IPython.display import Image, display
from airq.env import resolve_paths
from airq.pipelines.figures import rq4

tables = resolve_paths(cfg["paths"])["reports"] / "tables"
display(pd.read_csv(tables / "rq4_coverage.csv").round(3))
display(pd.read_csv(tables / "rq4_calibration.csv").round(3))
for p in rq4(cfg):
    if p.suffix == ".png":
        display(Image(filename=str(p)))


In [ ]:
# --- export: reports/ and models/ -> airq_artifacts_<date>.zip --------------------------
from airq.pipelines.colab import export_artifacts

zip_path = export_artifacts(cfg)
print("Wrote", zip_path.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
# locally:  python -m scripts.import_artifacts <zip>   (also runs the manifest check)
